# Qwen3 0.6B Fine-Tuning — Spam Classification (2007/2008 Dataset)

Fine-tunes `Qwen/Qwen3-0.6B` on the combined TREC-2007 + CEAS-2008 spam dataset using LoRA and `Qwen3ForSequenceClassification` (binary classification head, no text generation).

In [6]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from datasets import ClassLabel, DatasetDict, load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [10]:
MODEL_ID = "Qwen/Qwen3-0.6B"

SEED = 67
TRAIN_SPLIT = 0.90
VALIDATION_SPLIT = 0.05
TEST_SPLIT = 0.05
HOLDOUT_SPLIT = VALIDATION_SPLIT + TEST_SPLIT
MAX_SEQ_LENGTH = 448

# Tuned for an ~9 hour Apple Silicon run on an 18 GB MacBook Pro.
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 8e-5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.5

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MPS_MEMORY_FRACTION = 0.92

In [3]:
def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

DEVICE = find_device()
CUDA = DEVICE == "cuda"
IS_MPS = DEVICE == "mps"

if IS_MPS:
    if hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    if hasattr(torch.mps, "set_per_process_memory_fraction"):
        torch.mps.set_per_process_memory_fraction(MPS_MEMORY_FRACTION)
    print("⚠️ Using MPS acceleration on Apple Silicon.")
    if hasattr(torch.mps, "recommended_max_memory"):
        print(f"Recommended max MPS working set: {torch.mps.recommended_max_memory() / 1024**3:.2f} GB")
    print(f"MPS per-process memory fraction: {MPS_MEMORY_FRACTION:.2f}")
elif CUDA:
    print(f"✅ CUDA is available. Found {torch.cuda.device_count()} device(s).")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print("Will use bitsandbytes quantization.")
else:
    print("❌ Neither MPS nor CUDA is available. Falling back to CPU, which will be very slow.")

set_seed(SEED)

⚠️ Using MPS acceleration on Apple Silicon.
Recommended max MPS working set: 13.32 GB
MPS per-process memory fraction: 0.92


## Dataset

Load the pre-combined TREC-2007 + CEAS-2008 parquet file (15,000 rows, balanced 7500/7500).

In [4]:
DATASET_PATH = "../dataset/combined_datasets/spam_2007_2008/dataset.parquet"

dataset = load_dataset("parquet", data_files=DATASET_PATH, split="train")
dataset = dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))
print(f"Loaded {len(dataset)} rows")
print(f"Columns: {dataset.column_names}")
print(f"Spam: {sum(1 for s in dataset if s['label'] == 1)}, Ham: {sum(1 for s in dataset if s['label'] == 0)}")

Loaded 15000 rows
Columns: ['subject', 'body', 'label', 'source']
Spam: 7500, Ham: 7500


In [5]:
holdout = dataset.train_test_split(
    test_size=HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=TEST_SPLIT / HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)

dataset = DatasetDict({
    "train": holdout["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    spam = labels.count(1)
    ham = labels.count(0)
    print(f"{split}: {len(dataset[split])} rows — spam: {spam}, ham: {ham}")

train: 13500 rows — spam: 6750, ham: 6750
validation: 750 rows — spam: 375, ham: 375
test: 750 rows — spam: 375, ham: 375


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def dataset_transform(sample):
    subject = (sample["subject"] or "").strip()
    body = (sample["body"] or "").strip()
    parts = []
    if subject:
        parts.append(f"Subject: {subject}")
    if body:
        parts.append(body)
    sample["text"] = "\n\n".join(parts).strip()
    return sample

dataset = dataset.map(dataset_transform, desc="Building email texts")
dataset = dataset.remove_columns(["subject", "body", "source"])
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 13500
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 750
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 750
    })
})


## Model & LoRA

In [7]:
if CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
    )
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        dtype=torch.float32,
    )
if hasattr(model, "score"):
    model.score = model.score.to(torch.float32)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.problem_type = "single_label_classification"

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)
model = get_peft_model(model, peft_config)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()

trainable params: 10,094,592 || all params: 606,146,560 || trainable%: 1.6654


## Dataset Tokenization

Tokenize email text directly (no chat template) and rename `label` → `labels` for the Trainer.

In [9]:
def tokenize(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    encoded["length"] = [len(ids) for ids in encoded["input_ids"]]
    return encoded

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

rows_before = sum(split.num_rows for split in tokenized_dataset.values())

# ---> THE FIX: Filter out any sequences with a length of 0 <---
tokenized_dataset = tokenized_dataset.filter(
    lambda x: x["length"] > 0,
    desc="Filtering out empty sequences"
)

# Count total rows across all splits after filtering
rows_after = sum(split.num_rows for split in tokenized_dataset.values())

print(f"Removed {rows_before - rows_after} empty samples!")


tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
print(tokenized_dataset)

Removed 6 empty samples!
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 13494
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 750
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 750
    })
})


## Training

In [10]:
if CUDA:
    print("✅ CUDA detected: using paged_adamw_8bit + bf16.")
    target_optim = "paged_adamw_8bit"
else:
    print("⚠️ CUDA not available: using adamw_torch.")
    target_optim = "adamw_torch"

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
train_batches_per_epoch = int(np.ceil(len(tokenized_dataset["train"]) / TRAIN_BATCH_SIZE))
optimizer_steps_per_epoch = int(np.ceil(train_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS))
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch}")
print(f"Total optimizer steps: {optimizer_steps_per_epoch * NUM_TRAIN_EPOCHS}")

training_args = TrainingArguments(
    report_to="none",
    output_dir="./results/qwen3_0.6b_spam_mps",
    logging_strategy="steps",
    logging_steps=5,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="cosine",

    optim=target_optim,
    max_steps=-1,

    bf16=CUDA,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    group_by_length=True,
    length_column_name="length",

    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

⚠️ CUDA not available: using adamw_torch.
Max sequence length: 448
Effective batch size: 16
Optimizer steps per epoch: 844
Total optimizer steps: 1688


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = float((predictions == labels).mean())

    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    
    print({
        "accuracy": accuracy,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    })
    
    return {
        "accuracy": accuracy,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

NameError: name 'Trainer' is not defined

In [12]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.149000,0.040320,0.990667,0.986772,0.994667,0.990704
2,0.000000,0.057112,0.990667,0.991979,0.989333,0.990654


{'accuracy': 0.9906666666666667, 'precision': 0.9867724867724867, 'recall': 0.9946666666666667, 'f1': 0.9907038512616202}
{'accuracy': 0.9906666666666667, 'precision': 0.9919786096256684, 'recall': 0.9893333333333333, 'f1': 0.9906542056074765}


## Inference

In [28]:
import datetime

# Generates the timestamp in HHmmDDMMYYYY format (e.g., 105014042026)
timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
save_path = f"./results/qwen3_0.6b_spam_saved_weights_{timestamp}"

# Save the model and the tokenizer to the new timestamped directory
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model safely saved to disk at: {save_path}")

✅ Model safely saved to disk at: ./results/qwen3_0.6b_spam_saved_weights_105114042026


In [8]:
def _label_to_str(pred: int) -> str:
    return "spam" if pred == 1 else "valid"


def run_mail_classification(email_text: str) -> str:
    """Classify a raw email string as 'spam' or 'valid'."""
    email_text = email_text.strip()

    print(email_text)
    
    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenizer(
        email_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt"
    ).to(model_device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    return _label_to_str(pred)

def run_mail_from_dataset_classification(dataset, index: int) -> str:
    """Classify a sample from the dataset by its text field."""
    return run_mail_classification(dataset[index]["text"])

## Evaluation

In [ ]:
trainer.args.per_device_eval_batch_size = 1

test_predictions = trainer.predict(tokenized_dataset["test"])
test_pred_labels = np.argmax(test_predictions.predictions, axis=-1)
test_true_labels = np.array(tokenized_dataset["test"]["labels"])

rounded_metrics = {
    key: round(value, 4)
    for key, value in test_predictions.metrics.items()
    if isinstance(value, (int, float))
}
print(rounded_metrics)

test_dataset = dataset["test"]
incorrect_samples = []
for i, (pred, actual) in enumerate(zip(test_pred_labels, test_true_labels)):
    if pred != actual:
        incorrect_samples.append({
            "index": i,
            "content": test_dataset[i]["text"],
            "actual": _label_to_str(int(actual)),
            "output": _label_to_str(int(pred)),
        })


print(f"Test accuracy: {(test_pred_labels == test_true_labels).mean():.4f}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")

{'accuracy': 0.988, 'precision': 0.9790575916230366, 'recall': 0.9973333333333333, 'f1': 0.9881109643328929}
{'test_loss': 0.0328, 'test_accuracy': 0.988, 'test_precision': 0.9791, 'test_recall': 0.9973, 'test_f1': 0.9881, 'test_runtime': 109.9434, 'test_samples_per_second': 6.822, 'test_steps_per_second': 6.822}
Test accuracy: 0.4680
Mistakes: 399 / 750


In [37]:
trainer.args.group_by_length = False

# Now run predictions (the output will perfectly map 1:1 with your original dataset)
test_predictions = trainer.predict(tokenized_dataset["test"])
test_pred_labels = np.argmax(test_predictions.predictions, axis=-1)

# Now it is safe to pull the true labels straight from the dataset
test_true_labels = np.array(tokenized_dataset["test"]["labels"])

rounded_metrics = {
    key: round(value, 4)
    for key, value in test_predictions.metrics.items()
    if isinstance(value, (int, float))
}
print(rounded_metrics)

test_dataset = dataset["test"]
incorrect_samples = []
for i, (pred, actual) in enumerate(zip(test_pred_labels, test_true_labels)):
    if pred != actual:
        incorrect_samples.append({
            "index": i,
            "content": test_dataset[i]["text"],
            "actual": _label_to_str(int(actual)),
            "output": _label_to_str(int(pred)),
        })

print(f"Test accuracy: {(test_pred_labels == test_true_labels).mean():.4f}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")



{'accuracy': 0.988, 'precision': 0.9790575916230366, 'recall': 0.9973333333333333, 'f1': 0.9881109643328929}
{'test_loss': 0.0328, 'test_accuracy': 0.988, 'test_precision': 0.9791, 'test_recall': 0.9973, 'test_f1': 0.9881, 'test_runtime': 104.7232, 'test_samples_per_second': 7.162, 'test_steps_per_second': 7.162}
Test accuracy: 0.9880
Mistakes: 9 / 750


In [ ]:
print(incorrect_samples

In [11]:
print(run_mail_classification("""\
Subject: E-mail details of the client.
Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards, John
"""))

Subject: E-mail details of the client.
Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards, John
valid


In [12]:
print(run_mail_classification("""\
Subject: Free iPhone.
Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
"""))

Subject: Free iPhone.
Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
spam


In [16]:
print(run_mail_classification("""
Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the ‘Cards’ tab in the app to access your new card details and start making payments online.
"""))

Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the ‘Cards’ tab in the app to access your new card details and start making payments online.
spam


#### print(run_mail_classification("""\
Subject: Obsługa języka polskiego.
Kup najnowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym użytkownikiem!
Pozdrawiam, Wojciech
"""))

In [41]:
print("dupa")

dupa


In [51]:

print(run_mail_classification("""How are you doing? Can we meet later this afternoon? I just wanted to check if this information is correct."""))

spam
